[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/navidsharifi/optora-reproductions/blob/main/notebooks/reproductions.ipynb)

# Optora reproductions

Reproductions of published distributionally robust optimization papers with [Optora](https://github.com/navidsharifi/optora). This notebook is generated by `python -m tools.notebooks` from [`papers/`](https://github.com/navidsharifi/optora-reproductions/blob/main/papers) and [`results/`](https://github.com/navidsharifi/optora-reproductions/blob/main/results) of [navidsharifi/optora-reproductions](https://github.com/navidsharifi/optora-reproductions), so changes belong in those files, not here.

Papers in this notebook

- `duchi2019_variance_regularization`
- `ho2018_fast_bellman_updates`
- `mohajerin2018_wasserstein_dro`


## Recorded results

| Paper | Claim | Metric | Paper value | Ours, mean (std) | Seeds | Tolerance | Within tolerance | Device | optora |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| [duchi2019_variance_regularization](https://github.com/navidsharifi/optora-reproductions/blob/main/papers/duchi2019_variance_regularization/README.md) | c1 | fraction of replications with robust risk below ERM risk | 1 | 1 (0) | 3 | 0 | yes | NVIDIA GeForce RTX 3050 6GB Laptop GPU | 0.2.0 |
| [duchi2019_variance_regularization](https://github.com/navidsharifi/optora-reproductions/blob/main/papers/duchi2019_variance_regularization/README.md) | c2 | mean risk of the robust solution | 6.32e-18 | 0 (0) | 3 | 1e-08 | yes | NVIDIA GeForce RTX 3050 6GB Laptop GPU | 0.2.0 |
| [duchi2019_variance_regularization](https://github.com/navidsharifi/optora-reproductions/blob/main/papers/duchi2019_variance_regularization/README.md) | c3 | mean robust risk divided by mean ERM risk | 0.182 | 0.005475 (0.001256) | 3 | 0.05 | no | NVIDIA GeForce RTX 3050 6GB Laptop GPU | 0.2.0 |
| [ho2018_fast_bellman_updates](https://github.com/navidsharifi/optora-reproductions/blob/main/papers/ho2018_fast_bellman_updates/README.md) | c1 | max absolute error of q(xi) against the HiGHS linear program | 0 | 1.847e-13 (0) | 5 | 1e-09 | yes | Intel64 Family 6 Model 154 Stepping 3, GenuineIntel | 0.2.0 |
| [ho2018_fast_bellman_updates](https://github.com/navidsharifi/optora-reproductions/blob/main/papers/ho2018_fast_bellman_updates/README.md) | c2 | log10 of LP time over Optora time for one q(xi), S = 400 | 3 | 1.831 (0.008877) | 5 | 1 | no | Intel64 Family 6 Model 154 Stepping 3, GenuineIntel | 0.2.0 |
| [mohajerin2018_wasserstein_dro](https://github.com/navidsharifi/optora-reproductions/blob/main/papers/mohajerin2018_wasserstein_dro/README.md) | c1 | max_i \|x_N(eps)_i - 1/m\| at eps = 1, N = 300 | 0 | 1.621e-09 (3.644e-09) | 200 | 0.001 | yes | NVIDIA GeForce RTX 3050 6GB Laptop GPU | 0.3.0 |
| [mohajerin2018_wasserstein_dro](https://github.com/navidsharifi/optora-reproductions/blob/main/papers/mohajerin2018_wasserstein_dro/README.md) | c2 | indicator of J(x_N) <= J_N(eps) at eps = 1, N = 300 | 1 | 1 (0) | 200 | 0.02 | yes | NVIDIA GeForce RTX 3050 6GB Laptop GPU | 0.3.0 |
| [mohajerin2018_wasserstein_dro](https://github.com/navidsharifi/optora-reproductions/blob/main/papers/mohajerin2018_wasserstein_dro/README.md) | c3 | J at eps = 1e-4 minus J at the radius of lowest mean J, N = 30 | 0.2 | 0.154 (0.1184) | 200 | 0.1 | yes | NVIDIA GeForce RTX 3050 6GB Laptop GPU | 0.3.0 |

These are the committed records of the full runs. A claim is reproduced when the mean over seeds lies within the tolerance that was fixed before the run.


## Setup

Choose a GPU runtime (Runtime, Change runtime type), then run the cell below once before rerunning any paper.


In [ ]:
# Clone (or update) the repository and install it with its pinned optora
# release. Colab already ships a CUDA build of PyTorch, which pip keeps.
![ -d /content/optora-reproductions ] || git clone --depth 1 https://github.com/navidsharifi/optora-reproductions.git /content/optora-reproductions
!git -C /content/optora-reproductions pull --ff-only --quiet
%cd /content/optora-reproductions
!pip install --quiet -e .

import optora
import torch
from IPython.display import Markdown, display

from optora_reproductions.results import read_results
from tools.results_table import render_table

print("optora", optora.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name() if torch.cuda.is_available() else "none")


## Variance-based regularization with convex objectives

> Status. `partially reproduced`

### Reference

John Duchi and Hongseok Namkoong. Variance-based regularization with
convex objectives. Journal of Machine Learning Research, 20(68):1-55,
2019. arXiv:1610.02581. An earlier version appeared at NeurIPS 2017.

Citation checked against the JMLR page on 2026-11-15. The authors' public
code (`hsnamkoong/robustopt`) holds projection routines only, not the
simulation reproduced here.

### Claim

We reproduce the simulation of Section 5.2. The paper minimizes the
robustly regularized risk

$$
R_n(\theta, \mathcal P_n) = \sup_{P} \left\{ \mathbb E_P[\ell(\theta; X)] :
D_\phi(P \,\|\, \hat P_n) \le \rho / n \right\}, \qquad \phi(t) = \tfrac12 (t - 1)^2,
$$

and compares its minimizer $\hat\theta^{\rm rob}$ with the empirical risk
minimizer $\hat\theta^{\rm erm}$ through the population risk
$R(\theta) = \tfrac12 \|\theta - v\|_2^2$.

| Claim id | Statement |
| --- | --- |
| c1 | The robust solution has lower population risk than ERM in every replication, for every sample size and noise level |
| c2 | For $B = 1$ and $n = 1000$ the robust solution recovers $v$, so its risk is zero up to the numerical precision of the solver |
| c3 | For $B = 1$ and $n = 100$, the one setting where the robust solution is not $v$, its risk is about 0.18 times the ERM risk |

The location, metric, paper value and tolerance of each claim are fixed
in `claims.py`, committed before the full run.

### Setup

| Item | Value |
| --- | --- |
| Loss | $\ell(\theta; x) = \tfrac12 \lVert\theta - v\rVert_2^2 + x^\top(\theta - v)$ with $d = 50$ |
| Data | $X \sim \mathrm{Uni}(\{-B, B\}^d)$, $B \in \{0.01, 0.1, 1, 10\}$, $n \in \{10^2, 5\cdot10^2, 10^3, 5\cdot10^3, 10^4\}$ |
| Domain | $\Theta = \{\theta : \lVert\theta\rVert_2 \le r\}$, $r = 10$, $v = \tfrac{r}{2\sqrt d}\mathbf 1$ |
| Radius | $\rho = \log(2/\delta) + d \log(2nDL)$ (Eq. 17), $\delta = 0.05$, $D = 2r$, $L = 3r + \sqrt d B$ |
| Optora components | `optora.ChiSquareAmbiguitySet` with radius $2\rho/n$, since Optora's chi-square divergence $\sum_i (q_i - p_i)^2 / p_i$ equals $2 D_\phi$ |
| Robust solver | Two phases, batched over replications. First the solver of Section 5.1, projected gradient descent with a backtracking (Armijo) line search. Then a projected subgradient method with step $2/(k+1)$ and $k$-weighted averaging (Lacoste-Julien, Schmidt and Bach 2012), valid because $R_n$ is 1-strongly convex. Each replication keeps the point with the lower robust objective. Both phases take values and gradients from Optora only |
| ERM solver | Closed form $\hat\theta^{\rm erm} = \Pi_\Theta(v - \bar X_n)$, the exact minimizer of the empirical risk over $\Theta$ |
| Replications | 1,200 per $(n, B)$ as in the paper, split into 3 seeds of 400 so each claim has a spread over seeds |
| Precision | float64 |
| Hardware | RTX 3050 Laptop 6 GB (full run), CPU (smoke run) |

Known discrepancy, recorded before any run. Every ERM entry of Table 1
with $B \ge 0.1$ equals $dB^2/(12n)$ to about 1%. The setup as stated gives
$\mathbb E[R(\hat\theta^{\rm erm})] = \tfrac12 \mathbb E\lVert\bar X_n\rVert_2^2 = dB^2/(2n)$,
six times larger. We run the setup as stated. Claims c1 and c3 do not
depend on this scale and c2 sits at the numerical floor.

Solver change, made during the independent solver check (workflow step 6)
and before the full run. At $n = 100$ the radius $2\rho/n \approx 12$ lets
the chi-square ball hold every distribution on a set of about 8 tied top
losses, so the maximizer is not unique there and $R_n$ is not
differentiable. Section 5.1 states that $R_n$ is differentiable whenever
the losses have nonzero variance, but that does not hold in this regime.
The Section 5.1 solver stalls at such points: on 100 replications at
$n = 100$, $B = 1$ it gives a robust to ERM risk ratio of 0.519, while the
exact minimizer from an independent conic solver (CVXPY with Clarabel)
gives 0.006. The paper reports 0.182. Optora's worst-case value at the
conic optimum equals the conic objective to $10^{-6}$, and its gradient
matches finite differences, so the stall is a property of the solver, not
of Optora. With the user's approval the main result uses the two-phase
solver above, which matches the conic minimizer to within 1% in mean
risk. The descent phase alone is still recorded for every cell, so the
stall is reported next to it.

The subgradient phase runs only for replications whose descent point is
farther than $10^{-10}$ from $v$, and the table in the findings reports
that share per cell. A descent point at $v$ is kept as it is. This saves
hours of GPU time and is checked independently. In the solver check, the
descent phase ended at $v$ in all 400 replications at $n = 1000$,
$n = 5000$ and $n = 10^4$ (and in none at $n = 100$), and the conic solver
puts the minimizer at $v$ for every replication it solved at $n = 500$
(20 replications), $n = 5000$ and $n = 10^4$ (3 replications each, risk
below $10^{-18}$).

### Run

```bash
python -m papers.duchi2019_variance_regularization.run --config papers/duchi2019_variance_regularization/configs/smoke.toml
python -m papers.duchi2019_variance_regularization.run --config papers/duchi2019_variance_regularization/configs/full.toml
```

### Result

See the results table in the root `README.md`, generated from
`results/duchi2019_variance_regularization.<claim_id>.json`.

### Findings

Full run on 2026-11-15 with Optora 0.2.0, float64, on the RTX 3050. It took
3.8 hours of GPU time. The complete table is in
`runs/duchi2019_variance_regularization/full_table.md`, and every risk below
is a mean over the 1,200 replications of a cell.

| Claim | Paper | This run (mean over seeds, sample std) | Within tolerance |
| --- | --- | --- | --- |
| c1 | 1.0 | 1.0 (std 0) | yes |
| c2 | 6.32e-18 | 0 (std 0) | yes |
| c3 | 0.182 | 0.0055 (std 0.0013) | no |

Claims c1 and c2 reproduce. The robust solution beats ERM in every one of
the 24,000 replications, and for every $n \ge 500$ the descent phase stops at
$v$ itself, so no replication there needed the subgradient phase and the
mean risk is at most $2\cdot10^{-30}$. The paper's $6.32\cdot10^{-18}$ at
$n = 1000$ is consistent with the same point reached to a looser solver
tolerance.

Claim c3 does not reproduce, and the gap goes in the direction the solver
check predicted. At $n = 100$ the ratio of mean robust risk to mean ERM risk
is

| $B$ | Paper | Exact minimizer (two phases) | Descent phase only |
| --- | --- | --- | --- |
| 0.01 | 0.172-0.183 | 0.0057 | 0.516 |
| 0.1 | 0.172-0.183 | 0.0048 | 0.519 |
| 1 | 0.182 | 0.0055 | 0.522 |
| 10 | 0.172-0.183 | 0.0053 | 0.544 |

The ratio barely moves over three orders of magnitude in $B$, in either
solver column. The ERM scale gap below acts like a smaller $B$, so it is
unlikely to explain the difference. The
paper's value lies between the exact minimizer of $R_n$ and the point where
the Section 5.1 solver stalls. This fits an inexact solve of a nonsmooth
problem, but without the authors' simulation code we cannot pin down which
solver or stopping rule gives 0.18. The qualitative conclusion of Section
5.2 is stronger at the exact minimizer, with about 180 times less risk than
ERM at $n = 100$ instead of about 5 times.

The ERM column confirms the discrepancy recorded before the run. At
$B = 1$, $n = 100$ the mean ERM risk is 0.251, against $dB^2/(2n) = 0.25$ for
the stated setup and 0.042 in Table 1, and the same factor of six holds in
every cell with $B \ge 0.1$.

Optora. The chi-square worst case matched the conic reference and finite
differences throughout, and no run failed. The cost was the solve itself.
The $n = 100$ cells took about 600 seconds for 1,200 replications, most of
it in 5,000 subgradient steps on a $400 \times 100$ batch, where the
float64 tilt bisection of Optora 0.2.0 is bound by kernel launches. This
became Optora issue #108, fixed upstream by an exact sort-based chi-square
solve after this run. A rerun on a release that includes the fix is the
natural check of that change.


### Recorded results of `duchi2019_variance_regularization`

| Paper | Claim | Metric | Paper value | Ours, mean (std) | Seeds | Tolerance | Within tolerance | Device | optora |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| [duchi2019_variance_regularization](https://github.com/navidsharifi/optora-reproductions/blob/main/papers/duchi2019_variance_regularization/README.md) | c1 | fraction of replications with robust risk below ERM risk | 1 | 1 (0) | 3 | 0 | yes | NVIDIA GeForce RTX 3050 6GB Laptop GPU | 0.2.0 |
| [duchi2019_variance_regularization](https://github.com/navidsharifi/optora-reproductions/blob/main/papers/duchi2019_variance_regularization/README.md) | c2 | mean risk of the robust solution | 6.32e-18 | 0 (0) | 3 | 1e-08 | yes | NVIDIA GeForce RTX 3050 6GB Laptop GPU | 0.2.0 |
| [duchi2019_variance_regularization](https://github.com/navidsharifi/optora-reproductions/blob/main/papers/duchi2019_variance_regularization/README.md) | c3 | mean robust risk divided by mean ERM risk | 0.182 | 0.005475 (0.001256) | 3 | 0.05 | no | NVIDIA GeForce RTX 3050 6GB Laptop GPU | 0.2.0 |


### Rerun `duchi2019_variance_regularization` on Colab

The smoke configuration takes seconds. The full configuration repeats the recorded run and can take much longer.


In [ ]:
!python -m papers.duchi2019_variance_regularization.run --output-dir runs/colab --config papers/duchi2019_variance_regularization/configs/smoke.toml


In [ ]:
!python -m papers.duchi2019_variance_regularization.run --output-dir runs/colab --config papers/duchi2019_variance_regularization/configs/full.toml


In [ ]:
reruns = [r for r in read_results("runs/colab") if r.paper_id == "duchi2019_variance_regularization"]
display(Markdown(render_table(reruns)))


## Fast Bellman updates for robust MDPs

> Status. `partially reproduced`

### Reference

Chin Pang Ho, Marek Petrik and Wolfram Wiesemann. Fast Bellman updates for
robust MDPs. Proceedings of the 35th International Conference on Machine
Learning, PMLR 80:1979-1988, 2018.

Citation checked against the PMLR page on 2026-11-14. The authors' C++
implementation is linked from the paper; this reproduction does not use
it.

### Claim

The paper computes the robust Q-value of one state-action pair under an
$s,a$-rectangular $L_1$ ambiguity set,

$$
q(\xi) = \min_{p \in \Delta^S} \left\{ p^\top z : \|p - \bar p\|_1 \le \xi \right\},
$$

with a homotopy method, and compares its runtime with Gurobi. In Optora
this is one call of `TotalVariationAmbiguitySet`, because total variation
is half the $L_1$ distance:

$$
q(\xi) = -\sup_{p} \left\{ \mathbb E_p[-z] : \mathrm{TV}(p, \bar p) \le \xi / 2 \right\}.
$$

| Claim id | Statement |
| --- | --- |
| c1 | Optora returns the exact optimal value of the linear program, as the paper's methods do |
| c2 | Computing one $q(\xi)$ with Optora is about three orders of magnitude faster than with a general linear programming solver |

The location, metric, paper value and tolerance of each claim are fixed
in `claims.py`, committed before the full run.

### Setup

| Item | Value |
| --- | --- |
| Random domain | $\bar p$ drawn from $U[0, 1]^S$ and normalized, $z \sim U[0, 1]^S$ (rewards zero, values uniform), as in Section 5 |
| Inventory domain | Holding cost 0.1, purchase cost 1.0, sale price 1.6, no backlog, inventory capped at $S - 1$, demand rounded from $N(S/2, S/5)$, value function $v(s) = s$, initial state 0, every order quantity $a$ as one pair $(0, a)$, discount 0.9 (our choice; the paper does not state one) |
| Grid | $S \in \{50, 100, \dots, 400\}$, $\xi \in \{0, 0.25, \dots, 2\}$, 5 runs, as in Section 5.1 |
| Optora components | `optora.TotalVariationAmbiguitySet` with radius $\xi / 2$, batched over instances and budgets |
| Reference solver | HiGHS through `scipy.optimize.linprog`, on the linear program (6) of the paper |
| Timing (c2) | One $q(\xi)$ at $S = 400$ in the random domain, on CPU. Each timed call builds its problem from $(\bar p, z, \xi)$ and solves it, for both solvers. Median of repeated calls per budget, averaged over the nine budgets and the five runs, as the paper averages over $\kappa$ |
| Precision | float64 |
| Hardware | The laptop CPU recorded in each result's environment (Intel64 family 6 model 154), with the machine otherwise idle during the full run |
| Differences from the paper | HiGHS instead of Gurobi 7.5, Python and PyTorch instead of C++, other hardware. Only the unweighted $L_1$ norm is reproduced: a weighted $L_1$ ball is not a $\varphi$-divergence ball and no shipped Optora set covers it. The $s$-rectangular bisection of Section 5.2 couples the actions of a state and is not one ambiguity-set call, so it is out of scope |

### Run

```bash
python -m papers.ho2018_fast_bellman_updates.run --config papers/ho2018_fast_bellman_updates/configs/smoke.toml
python -m papers.ho2018_fast_bellman_updates.run --config papers/ho2018_fast_bellman_updates/configs/full.toml
```

### Result

See the results table in the root `README.md`, generated from
`results/ho2018_fast_bellman_updates.<claim_id>.json`.

### Findings

Full run on 2026-11-15 with Optora 0.2.0, float64, on the laptop CPU with
nothing else heavy running. It took about three minutes. The complete table
is in `runs/ho2018_fast_bellman_updates/full_table.md`.

| Claim | Paper | This run (mean over 5 runs, sample std) | Within tolerance |
| --- | --- | --- | --- |
| c1 | 0 | 1.8e-13 (std 0) | yes |
| c2 | 3.0 | 1.83 (std 0.009) | no |

Claim c1 reproduces. Over every size, budget and run, Optora's $q(\xi)$
matches the HiGHS linear program to $6.7\cdot10^{-16}$ in the random domain
and $1.9\cdot10^{-13}$ in the inventory domain. The inventory error grows
with $S$ only because the values $z$ grow with $S$ (to a few hundred), so it
stays within a few units in the last place.

Claim c2 does not reproduce. One $q(\xi)$ at $S = 400$ takes 7.4 ms with
HiGHS and 0.109 ms with Optora, a speedup of about 68 rather than about
1,000.

| $S$ | HiGHS ms | Optora ms | Speedup |
| --- | --- | --- | --- |
| 50 | 2.20 | 0.092 | 24 |
| 200 | 4.00 | 0.101 | 40 |
| 400 | 7.42 | 0.109 | 68 |

The HiGHS time grows with $S$ while the Optora time barely moves, so a
single Optora call is bound by the fixed cost of about fifteen small eager
PyTorch operations, not by the $O(S \log S)$ sort. The paper times a C++
implementation against Gurobi, so its ratio measures a different pair of
programs, and the gap here is about per-call overhead rather than the
algorithm.

A post hoc measurement, made after the run and not part of any claim,
shows what batching does. Evaluating 400 instances at all nine budgets in
one call costs 22.7 ms on the same CPU, about 6 $\mu$s per $q(\xi)$, which
is roughly 1,200 times faster than one HiGHS solve at $S = 400$. This is
how a robust value iteration would call Optora, with every state-action
pair in one batch, but the pre-registered claim times a single call and
stays outside its tolerance.

Optora. No mismatch or failure. The per-call overhead was profiled earlier
(about 15 eager operations of 3-5 $\mu$s each and one sort) and is normal
for eager PyTorch, so it is recorded here rather than filed.


### Recorded results of `ho2018_fast_bellman_updates`

| Paper | Claim | Metric | Paper value | Ours, mean (std) | Seeds | Tolerance | Within tolerance | Device | optora |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| [ho2018_fast_bellman_updates](https://github.com/navidsharifi/optora-reproductions/blob/main/papers/ho2018_fast_bellman_updates/README.md) | c1 | max absolute error of q(xi) against the HiGHS linear program | 0 | 1.847e-13 (0) | 5 | 1e-09 | yes | Intel64 Family 6 Model 154 Stepping 3, GenuineIntel | 0.2.0 |
| [ho2018_fast_bellman_updates](https://github.com/navidsharifi/optora-reproductions/blob/main/papers/ho2018_fast_bellman_updates/README.md) | c2 | log10 of LP time over Optora time for one q(xi), S = 400 | 3 | 1.831 (0.008877) | 5 | 1 | no | Intel64 Family 6 Model 154 Stepping 3, GenuineIntel | 0.2.0 |


### Rerun `ho2018_fast_bellman_updates` on Colab

The smoke configuration takes seconds. The full configuration repeats the recorded run and can take much longer.


In [ ]:
!python -m papers.ho2018_fast_bellman_updates.run --output-dir runs/colab --config papers/ho2018_fast_bellman_updates/configs/smoke.toml


In [ ]:
!python -m papers.ho2018_fast_bellman_updates.run --output-dir runs/colab --config papers/ho2018_fast_bellman_updates/configs/full.toml


In [ ]:
reruns = [r for r in read_results("runs/colab") if r.paper_id == "ho2018_fast_bellman_updates"]
display(Markdown(render_table(reruns)))


## Data-driven distributionally robust optimization using the Wasserstein metric

> Status. `reproduced`

### Reference

Peyman Mohajerin Esfahani and Daniel Kuhn. Data-driven distributionally
robust optimization using the Wasserstein metric: performance guarantees
and tractable reformulations. Mathematical Programming, 171(1-2):115-166,
2018. doi:10.1007/s10107-017-1172-1. arXiv:1505.05116.

Citation checked against the published version on 2026-11-15. Section
numbers below follow arXiv v3. No public code accompanies the paper.

### Claim

We reproduce the mean-risk portfolio experiment of Section 7.2.A. For a
training dataset $\hat\Xi_N$ of $N$ return vectors, the paper solves

$$
\hat J_N(\varepsilon) = \inf_{x \in \mathbb X,\, \tau \in \mathbb R}\;
\sup_{Q \in \mathbb B_\varepsilon(\hat P_N)}
\mathbb E_Q\left[\max_{k \le 2} a_k \langle x, \xi\rangle + b_k \tau\right],
$$

where $\mathbb B_\varepsilon(\hat P_N)$ is the 1-Wasserstein ball of
radius $\varepsilon$ around the empirical distribution and $\mathbb X$ is
the probability simplex (Eq. 50). The minimizer $\hat x_N(\varepsilon)$ has
out-of-sample performance $J(\hat x_N(\varepsilon))$, the mean-CVaR
objective under the true distribution, and $\hat J_N(\varepsilon)$
certifies it when $J(\hat x_N(\varepsilon)) \le \hat J_N(\varepsilon)$.

| Claim id | Statement |
| --- | --- |
| c1 | For a large radius the robust portfolio is the equally weighted one (Proposition 7.2, Figure 4) |
| c2 | At $\varepsilon = 1$ and $N = 300$ the certificate holds in every simulation, so the reliability is one (Figure 5(b)) |
| c3 | With $N = 30$ samples a well chosen radius improves the out-of-sample performance over the nearly nonrobust radius $10^{-4}$ by about 0.2 (Figure 5(a)) |

The location, metric, paper value and tolerance of each claim are fixed
in `claims.py`, committed before the full run. The values of c2 and c3 are
read from the figures. The tolerance of c3 is loose for that reason.

### Setup

| Item | Value |
| --- | --- |
| Returns | $m = 10$ assets, $\xi_i = \psi + \zeta_i$ with $\psi \sim N(0, 0.02^2)$ and $\zeta_i \sim N(0.03\,i, (0.025\,i)^2)$ independent (Section 7.2) |
| Loss | $a = (-1, -1 - \rho/\alpha)$, $b = (\rho, \rho(1 - 1/\alpha))$, $\alpha = 0.2$, $\rho = 10$ (Section 7.1) |
| Ambiguity set | 1-Wasserstein ball with the 1-norm on $\mathbb R^m$, support $\Xi = \mathbb R^m$ (Section 7.2) |
| Out-of-sample $J$ | Closed form under the true normal distribution, $J(x) = -(1 + \rho)\mu^\top x + \frac{\rho}{\alpha}\varphi(\Phi^{-1}(\alpha))\sqrt{x^\top \Sigma x}$, as Section 7.2.A computes it analytically |
| Simulations | 200 independent training datasets per sample size, one seed each (seeds 0-199) |
| c1, c2 | $N = 300$, $\varepsilon = 1$ |
| c3 | $N = 30$, $\varepsilon \in \{b \cdot 10^c : b \in \{1, \dots, 9\}, c \in \{-4, \dots, -1\}\} \cup \{1\}$, 37 radii |
| Optora components | `optora.WassersteinAmbiguitySet` with one batched ground cost of shape `(200, 1, n, n)` and the radii as a batch (Optora 0.3.0, #112) |
| Finite support | The $N$ samples and the $2m$ points $\pm D e_j$, $D = 10^6$, with nominal mass $1/N$ on the samples and 0 on the far points, cost $\lVert\xi - \xi'\rVert_1$. By Corollary 5.1 the worst case over $\mathbb R^m$ is $\frac1N\sum_i \ell(\hat\xi_i) + \varepsilon \max_k \lvert a_k\rvert \lVert x\rVert_\infty$, a supremum that is approached but not attained as mass $\varepsilon/D$ moves to distance $D$. This support gives the same value up to $O(1/D)$, which was checked against the closed form and the linear program to about $10^{-8}$ |
| Robust solver | Central-cut ellipsoid method (Bland, Goldfarb and Todd 1981) on $(y, \tau) \in \mathbb R^{10}$ with $x = e_{10} + U y$, where $U$ is an orthonormal basis of $\{\mathbf 1^\top x = 0\}$, batched over simulations and radii. Values and subgradients come from Optora only, and a centre with some $x_j < 0$ gets the cut $-\nabla x_j$. The first ellipsoid is centred at the portfolio fully invested in asset 10 with $\tau = 0$, away from the answer of c1, and contains the simplex times $[-\tau_{\max}, \tau_{\max}]$ with $\tau_{\max} = \max_{i,j} \lvert\hat\xi_{ij}\rvert$, which holds an optimal $\tau$ because the radius term of Corollary 5.1 does not depend on $\tau$. The output is the feasible centre of lowest value after 3000 steps |
| Certificate | $\hat J_N(\varepsilon)$ is Optora's worst-case value at the returned $(\hat x, \hat\tau)$. For this pair $J(\hat x) \le \mathbb E_P[\ell(\hat x, \hat\tau, \xi)]$, which is at most the worst case over the ball whenever $P$ lies in it, the event behind Theorem 3.4. The finite support changes that worst case by $O(1/D)$ only, so c2 tests the same event |
| Reference solver | The linear program of Theorem 4.2 with $\Xi = \mathbb R^m$, solved by HiGHS through `scipy.optimize.linprog` for every simulation and radius. It is reported next to the results and is never the source of a claim value |
| Precision | float64 |
| Hardware | RTX 3050 Laptop 6 GB (full run), CPU (smoke run) |

Choices recorded before any full run.

- The paper writes the moments as $N(0, 2\%)$ and $N(i \times 3\%, i \times 2.5\%)$ without saying whether the second argument is a variance or a standard deviation. Reading it as a standard deviation puts $J$ of the equally weighted portfolio at about $-1.07$, on the scale of Figure 5. Reading it as a variance gives about $+0.76$, which does not fit the figure, so we use standard deviations.
- Section 7.2.B selects radii from $\{b \cdot 10^c : b \in \{0, \dots, 9\}, c \in \{-3, -2, -1\}\}$, but the grid of Figure 5 is not stated. Our sweep covers the horizontal range of Figure 5, from $10^{-4}$ to 1.
- The panels for $N = 3000$ are not reproduced. One batched cost would hold $200 \cdot 3020^2$ float64 entries (about 14.6 GB), and no claim needs them.
- The paper solves the convex reformulation. Here an oracle method on Optora's values is used instead, so that Optora is what produces every claim value. The tolerance of c1 is $10^{-3}$ rather than solver precision for that reason.

Solver change, made during the independent solver check (workflow step 6) and before the full run. The approved plan named a projected subgradient method with step $0.05/\sqrt k$ and averaged iterates. On 8 simulations it left $J(\hat x)$ up to 0.2 away from $J$ of the HiGHS solution at $\varepsilon = 10^{-4}$, $N = 30$, after 5000 steps, which is the size of the effect c3 measures. The outer problem has only 10 variables, so the ellipsoid method fits better. Its volume shrinks by at least $e^{-1/22}$ per step, and on the same 8 simulations after 2000 steps it put $J(\hat x)$ within $3 \cdot 10^{-4}$ of the HiGHS solution's $J$ and $\hat J_N$ within $2 \cdot 10^{-5}$ of the optimal value at every tested radius. Both methods use the same oracle, Optora's value and subgradient.

### Run

```bash
python -m papers.mohajerin2018_wasserstein_dro.run --config papers/mohajerin2018_wasserstein_dro/configs/smoke.toml
python -m papers.mohajerin2018_wasserstein_dro.run --config papers/mohajerin2018_wasserstein_dro/configs/full.toml
```

### Result

See the results table in the root `README.md`, generated from
`results/mohajerin2018_wasserstein_dro.<claim_id>.json`.

### Findings

Full run on 2026-11-15 with Optora 0.3.0, float64, on the RTX 3050. It took
2 hours 12 minutes, about half for each sample size. The complete table is
in `runs/mohajerin2018_wasserstein_dro/full_table.md`.

| Claim | Paper | This run (mean over 200 simulations, sample std) | Within tolerance |
| --- | --- | --- | --- |
| c1 | 0 | 1.6e-9 (std 3.6e-9) | yes |
| c2 | 1.0 | 1.0 (std 0) | yes |
| c3 | 0.2 | 0.154 (std 0.118) | yes |

All three claims reproduce. At $\varepsilon = 1$ and $N = 300$ the robust
portfolio is the equally weighted one in every simulation, with the largest
weight error $3.7\cdot10^{-8}$. HiGHS returns exactly $1/m$ there, and the
largest distance between our portfolio and the HiGHS one is
$3.8\cdot10^{-8}$. The certificate holds in all 200 simulations, but this
is an easy test at this radius. By Corollary 5.1 the radius term alone adds
$\varepsilon \max_k \lvert a_k\rvert \lVert x\rVert_\infty = 5.1$ to the
empirical mean, while $J$ of the equally weighted portfolio is $-1.07$.

The sweep at $N = 30$ shows the shape of Figure 5(a).

| $\varepsilon$ | Mean $J$ | 20% quantile | 80% quantile | Reliability |
| --- | --- | --- | --- | --- |
| $10^{-4}$ | -1.098 | -1.213 | -1.000 | 0.010 |
| $10^{-3}$ | -1.106 | -1.222 | -1.005 | 0.010 |
| $10^{-2}$ | -1.172 | -1.260 | -1.076 | 0.105 |
| 0.03 | -1.236 | -1.302 | -1.181 | 0.485 |
| 0.05 | -1.252 | -1.312 | -1.211 | 0.900 |
| 0.07 | -1.244 | -1.302 | -1.202 | 1.000 |
| 0.1 | -1.214 | -1.268 | -1.167 | 1.000 |
| 0.3 | -1.073 | -1.074 | -1.074 | 1.000 |
| 1 | -1.074 | -1.074 | -1.074 | 1.000 |

The mean $J$ falls until $\varepsilon = 0.05$ and rises after it. From
$\varepsilon = 0.3$ on, every simulation returns the equally weighted
portfolio, so the 20% and 80% quantiles coincide at $-1.074$. The
difference c3 measures is 0.154 on average and positive in 191 of the 200
simulations. Both mean levels lie a little below the values read from the
figure (about $-1.0$ at $10^{-4}$ and $-1.2$ at the minimum), which fits a
reading off a plot, and the difference between them stays inside the
tolerance. The reliability reaches one at $\varepsilon = 0.07$, slightly
above the radius of best mean performance.

Every certificate is within $1.6\cdot10^{-6}$ of the HiGHS optimal value,
at every radius and both sample sizes. So the ellipsoid method on Optora's
values solves the robust problem to well below the effects the claims
measure.

Optora. No mismatch or failure. The batched ground cost of #112 let one
`WassersteinAmbiguitySet` call cover 200 simulations and 37 radii, with a
peak of 0.81 GB on the GPU. The limit was the dense $n \times n$ cost,
which rules out the $N = 3000$ panels (about 14.6 GB). A dense cost matrix
is the standard input of discrete optimal transport, so this is recorded
here rather than filed.


### Recorded results of `mohajerin2018_wasserstein_dro`

| Paper | Claim | Metric | Paper value | Ours, mean (std) | Seeds | Tolerance | Within tolerance | Device | optora |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| [mohajerin2018_wasserstein_dro](https://github.com/navidsharifi/optora-reproductions/blob/main/papers/mohajerin2018_wasserstein_dro/README.md) | c1 | max_i \|x_N(eps)_i - 1/m\| at eps = 1, N = 300 | 0 | 1.621e-09 (3.644e-09) | 200 | 0.001 | yes | NVIDIA GeForce RTX 3050 6GB Laptop GPU | 0.3.0 |
| [mohajerin2018_wasserstein_dro](https://github.com/navidsharifi/optora-reproductions/blob/main/papers/mohajerin2018_wasserstein_dro/README.md) | c2 | indicator of J(x_N) <= J_N(eps) at eps = 1, N = 300 | 1 | 1 (0) | 200 | 0.02 | yes | NVIDIA GeForce RTX 3050 6GB Laptop GPU | 0.3.0 |
| [mohajerin2018_wasserstein_dro](https://github.com/navidsharifi/optora-reproductions/blob/main/papers/mohajerin2018_wasserstein_dro/README.md) | c3 | J at eps = 1e-4 minus J at the radius of lowest mean J, N = 30 | 0.2 | 0.154 (0.1184) | 200 | 0.1 | yes | NVIDIA GeForce RTX 3050 6GB Laptop GPU | 0.3.0 |


### Rerun `mohajerin2018_wasserstein_dro` on Colab

The smoke configuration takes seconds. The full configuration repeats the recorded run and can take much longer.


In [ ]:
!python -m papers.mohajerin2018_wasserstein_dro.run --output-dir runs/colab --config papers/mohajerin2018_wasserstein_dro/configs/smoke.toml


In [ ]:
!python -m papers.mohajerin2018_wasserstein_dro.run --output-dir runs/colab --config papers/mohajerin2018_wasserstein_dro/configs/full.toml


In [ ]:
reruns = [r for r in read_results("runs/colab") if r.paper_id == "mohajerin2018_wasserstein_dro"]
display(Markdown(render_table(reruns)))


## All rerun results

Every claim rerun in this session.


In [ ]:
reruns = [r for r in read_results("runs/colab")]
display(Markdown(render_table(reruns)))
